In [1]:
import sys
print(sys.executable)

/Users/joelbenson/Desktop/SmartphoneAddictionPrediction/.venv/bin/python3


In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("data/train.csv")
print(df.shape)
print(df.columns.tolist())

(691369, 14)
['id', 'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'addicted_label']


In [4]:
print(df.dtypes)
print(df.head().T)

id                           int64
age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
gender                         str
stress_level                   str
academic_work_impact           str
addicted_label               int64
dtype: object
                              0       1       2      3       4
id                            0       1       2      3       4
age                        24.0    19.0    18.0   21.0    26.0
daily_screen_time_hours     NaN    5.97    5.09   6.42    11.2
social_media_hours         1.83    1.08     NaN   1.26    1.87
gaming_hours               1.59     NaN     NaN   1.42    2.81
work_study_hours           2.11    3.03     NaN   3.36    1.95
sleep_hours                7.46    8.22    6.25   8.85 

In [5]:
missing = df.isna().mean().mul(100).round(2)
print("% Values Missing---------")
print(missing[missing > 0].sort_values(ascending=False))

% Values Missing---------
social_media_hours         19.38
gaming_hours               18.34
weekend_screen_time        16.21
daily_screen_time_hours    13.86
app_opens_per_day          11.67
notifications_per_day       9.78
stress_level                7.98
work_study_hours            7.45
sleep_hours                 6.43
academic_work_impact        6.40
gender                      4.20
age                         4.18
dtype: float64


In [6]:
print(df["addicted_label"].value_counts())
print(df["addicted_label"].value_counts(normalize=True).round(4))

addicted_label
1    490474
0    200895
Name: count, dtype: int64
addicted_label
1    0.7094
0    0.2906
Name: proportion, dtype: float64


In [7]:
num_cols = df.select_dtypes(include="number").columns.drop(["id", "addicted_label"])
print(df[num_cols].describe().T)

                            count        mean        std    min    25%  \
age                      662440.0   26.615408   5.153162  18.00  22.00   
daily_screen_time_hours  595515.0    7.640865   2.721446   0.50   5.48   
social_media_hours       557374.0    2.471038   1.316137   0.00   1.45   
gaming_hours             564548.0    1.459265   0.934552   0.00   0.70   
work_study_hours         639851.0    2.366971   1.258797   0.00   1.36   
sleep_hours              646889.0    6.804334   1.234512   4.50   5.78   
notifications_per_day    623785.0  145.894900  65.917556  20.00  93.00   
app_opens_per_day        610659.0  102.636781  48.093970  15.00  64.00   
weekend_screen_time      579306.0    9.479866   2.856006   0.51   7.28   

                            50%     75%     max  
age                       27.00   31.00   35.00  
daily_screen_time_hours    7.77    9.84   15.00  
social_media_hours         2.31    3.37    8.00  
gaming_hours               1.33    2.09    4.00  
work_stud

In [8]:
for col in ["gender", "stress_level", "academic_work_impact"]:
    print(df[col].value_counts(dropna=False))
    print()

gender
Male      223662
Female    221595
Other     217078
NaN        29034
Name: count, dtype: int64

stress_level
High      220873
Low       207783
Medium    207565
NaN        55148
Name: count, dtype: int64

academic_work_impact
Yes    330566
No     316579
NaN     44224
Name: count, dtype: int64



In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["id", "addicted_label"]).copy()
y = df["addicted_label"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
from sklearn.impute import SimpleImputer

cat_cols = ["gender", "stress_level", "academic_work_impact"]
num_cols = X.select_dtypes("number").columns.tolist()

num_imputer = SimpleImputer(strategy="median")
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_valid[num_cols] = num_imputer.transform(X_valid[num_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_valid[cat_cols] = cat_imputer.transform(X_valid[cat_cols])

In [13]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_valid_cat = encoder.transform(X_valid[cat_cols])

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_valid_num = scaler.transform(X_valid[num_cols])

In [15]:
X_train_final = np.hstack([X_train_num, X_train_cat])
X_valid_final = np.hstack([X_valid_num, X_valid_cat])

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_final, y_train)

valid_proba = model.predict_proba(X_valid_final)[:, 1]
valid_pred = model.predict(X_valid_final)

print("AUC     :", roc_auc_score(y_valid, valid_proba))
print("Accuracy:", accuracy_score(y_valid, valid_pred))
print("F1      :", f1_score(y_valid, valid_pred))

AUC     : 0.9107548410196142
Accuracy: 0.8331067301155677
F1      : 0.875702228278726
